## 1 — Imports


---


## 2 — JSON — `json.loads()` and `json.dumps()`

| Function | Direction | When to use |
|----------|-----------|-------------|
| `json.loads(string)` | string → Python dict | Parse LLM response |
| `json.dumps(dict)` | Python dict → string | Write log entries |
| `json.load(file)` | open file → Python dict | Load a `.json` config file |
| `json.dump(dict, file)` | Python dict → open file | Save results to `.json` |

> **The LLM API always returns a string** — even when the LLM was told to return JSON.  
> You must call `json.loads()` before you can access any field.


In [4]:
# ── json.dumps() — dict → string ───────────────────────────
config = {'model': 'gpt-4o', 'max_tokens': 1024, 'temperature': 0.2}

compact = json.dumps(config)           # single line — for logs
pretty  = json.dumps(config, indent=2) # indented — for .json files

print('Compact:', compact)
print('Type   :', type(compact))   # str
print()
print('Pretty:')
print(pretty)


Compact: {"model": "gpt-4o", "max_tokens": 1024, "temperature": 0.2}
Type   : <class 'str'>

Pretty:
{
  "model": "gpt-4o",
  "max_tokens": 1024,
  "temperature": 0.2
}


In [6]:
# ── json.load() and json.dump() — file versions ─────────────

# json.load  → read a dict directly from an open file

with open('model_config.json', 'r', encoding='utf-8') as f:
    loaded = json.load(f)

print('Loaded back:', loaded)
print('Type       :', type(loaded).__name__)   # dict


Loaded back: {'model': 'gpt-4o', 'max_tokens': 1024, 'temperature': 0.2}
Type       : dict


## 3 — The `with` Statement — Safe File Handling

The `with` statement guarantees the file is **closed automatically** — even if an error occurs inside the block.

```python
# Without with — risky:
f    = open('file.txt')
data = f.read()    # if this raises an error, f.close() is NEVER called
f.close()

# With the with statement — safe:
with open('file.txt') as f:
    data = f.read()    # file is always closed when the block exits
```

| Mode | String | Behaviour |
|------|--------|-----------|
| Read | `'r'` | Open existing file. `FileNotFoundError` if missing. |
| Write | `'w'` | Create file. **Overwrites** if already exists. |
| Append | `'a'` | Add to the end. Creates file if missing. |

> Always add `encoding='utf-8'` to avoid silent text corruption on Windows.


In [15]:
def load_system_prompt(filepath='sample_prompt.txt'):
    """
    Load a system prompt from a .txt file.
    Keeping prompts in files means the team can edit them without touching Python.
    """
    # 'r' = read mode  |  encoding='utf-8' = always specify
    # with block closes the file automatically — even if f.read() raises an error
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()   # loads the entire file as one string

    # .strip() removes the trailing newline that text editors always add
    return content.strip()


In [16]:
# Now call the function — reads the file back
prompt = load_system_prompt('sample_prompt.txt')
print(f'Loaded {len(prompt)} characters:')
print(prompt)

Loaded 112 characters:
## Role
You are a customer support assistant for ShopSmart.

## Task
Answer using ONLY the information provided.


In [17]:
def save_llm_response_log(query, response, log_file='llm_calls.log'):
    """
    Append one LLM call to a JSONL log file.
    JSONL = JSON Lines: one JSON object per line.
    """
    # 'a' = append mode:
    #   file exists  → adds to the END, does not overwrite previous entries
    #   file missing → creates it automatically
    with open(log_file, 'a', encoding='utf-8') as f:
        # json.dumps() converts a Python dict → JSON string
        entry = json.dumps({'query': query, 'response': response})
        f.write(entry + '\n')   # one entry per line = JSONL format


In [18]:
save_llm_response_log('Where is my order?', 'Your order ORD-3042 is In Transit.')

In [20]:
# Read back every line — same filename the function wrote to
# TRACE:
# line 0 → '{"query": "Where is my order?", "response": "..."}\n'
#           .strip() removes \n  →  json.loads() converts string → dict
# line 1 → second entry, same process
print('Log contents:')
with open('llm_calls.log', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        record = json.loads(line.strip())
        print(f'  [{i+1}] Q: {record["query"]}')
        print(f'       A: {record["response"]}')

Log contents:
  [1] Q: Where is my order?
       A: Your order ORD-3042 is In Transit.
  [2] Q: Can I return this?
       A: Returns accepted within 30 days.


### `parse_llm_json_response()` — handles plain and fenced JSON


In [39]:
# ── Demo 1: plain JSON string ────────────────────────────────
raw = '{"category": "TRACK_ORDER", "confidence": "high"}'

print('Before parse — type:', type(raw).__name__)   # str
parsed = parse_llm_json_response(raw)
print('After parse  — type:', type(parsed).__name__) # dict
print('category  :', parsed['category'])
print('confidence:', parsed['confidence'])


Before parse — type: str
After parse  — type: dict
category  : TRACK_ORDER
confidence: high


---


### Reading customer rows from CSV


In [55]:
for c in customers:
    print(f"  {c['customer_id']}  {c['name']:20s}  {c['city']}, {c['state']}")

print()
print('--- CSV type trap ---')
print('customer_id type    :', type(customers[0]['customer_id']).__name__, '← str!')
print("String compare trap : '1001' > '200' =", '1001' > '200', '← WRONG (alphabetic)')
print('Correct int compare : 1001  > 200  =',   1001  > 200)
print('Fix                 :', int(customers[0]['customer_id']))


  1001  Prudhvi Akella        Rajahmundry, AP
  1002  Ravi Kumar            Hyderabad, TS
  1003  Ananya Sharma         Bangalore, KA

--- CSV type trap ---
customer_id type    : str ← str!
String compare trap : '1001' > '200' = False ← WRONG (alphabetic)
Correct int compare : 1001  > 200  = True
Fix                 : 1001


In [56]:
# ── Semicolon-delimited (common in European exports) ────────
semicolon_csv = """customer_id;name;city
1001;Prudhvi Akella;Rajahmundry
1002;Ravi Kumar;Hyderabad
"""

reader = csv.DictReader(io.StringIO(semicolon_csv), delimiter=';')
for row in reader:
    print(row)


{'customer_id': '1001', 'name': 'Prudhvi Akella', 'city': 'Rajahmundry'}
{'customer_id': '1002', 'name': 'Ravi Kumar', 'city': 'Hyderabad'}


In [58]:
# ── Pipe-separated ──────────────────────────────────────────
pipe_csv = """customer_id|name|city
1001|Prudhvi Akella|Rajahmundry
1002|Ravi Kumar|Hyderabad
"""

reader = csv.DictReader(io.StringIO(pipe_csv), delimiter='|')
for row in reader:
    print(row)


{'customer_id': '1001', 'name': 'Prudhvi Akella', 'city': 'Rajahmundry'}
{'customer_id': '1002', 'name': 'Ravi Kumar', 'city': 'Hyderabad'}


In [59]:
# No header row — supply column names with fieldnames=
no_header_csv = """1001,Prudhvi Akella,Rajahmundry
1002,Ravi Kumar,Hyderabad
"""

reader = csv.DictReader(
    io.StringIO(no_header_csv),
    fieldnames=['customer_id', 'name', 'city']   # supply headers manually
)
for row in reader:
    print(row)


{'customer_id': '1001', 'name': 'Prudhvi Akella', 'city': 'Rajahmundry'}
{'customer_id': '1002', 'name': 'Ravi Kumar', 'city': 'Hyderabad'}


In [78]:
# csv.Sniffer — auto-detect delimiter
sample = """customer_id,name,email,city,state
1001,Prudhvi Akella,"prudhvi@example.com,prudhvi1@example2.com",Rajahmundry,AP
1002,Ravi Kumar,ravi@example.com,Hyderabad,TS
1003,Ananya Sharma,ananya@example.com,Bangalore,KA
"""

dialect  = csv.Sniffer().sniff(sample)   # sniff() reads a sample and guesses
print('Detected delimiter:', repr(dialect.delimiter))
print("Quote character:", repr(dialect.quotechar))
print(dialect)

<class 'type'>
Detected delimiter: ','
Quote character: '"'


---


### Step 1 — Load the system prompt from a file


### Step 2 — Read customer data


### Step 3 — Build the user message with an f-string


### Step 4 — Receive LLM response (simulated with fences)


### Step 5 — Parse the response


### Step 6 — Log the call


---
